# Quick Expeirment Demo

In [ ]:
import json

import pandas as pd
from plotnine import aes, geom_point, ggplot, labs, theme, theme_bw

from dataset_similarity.constants import PROJECT_DIR
from dataset_similarity.utils import load_yaml_from_path

In [ ]:
CFG_DIR = PROJECT_DIR / "configs"
RESUTLS_DIR = PROJECT_DIR / "results"
METRICS_DIR = RESUTLS_DIR / "metrics"
EVAL_DIR = RESUTLS_DIR / "eval"

In [ ]:
experiment = "experiment_1_qe"
experiment_cfg = load_yaml_from_path(CFG_DIR / "experiments" / f"{experiment}.yaml")

In [ ]:
# Load eval results
def load_eval_results(dataset_name):
    dataset_name = dataset_name.replace("test", "train")
    result_path = EVAL_DIR / f"{dataset_name}_results.json"
    with open(result_path) as f:
        return json.load(f)
eval_results = pd.DataFrame(
    load_eval_results(dataset_name) for dataset_name in experiment_cfg["datasets"]
)

In [ ]:
# Load metrics results
def load_json(json_path):
    with open(json_path) as f:
        return json.load(f)

metrics_results = pd.DataFrame(
    load_json(result) for result in (METRICS_DIR / experiment).glob("*.json")
)

## Plot Fine-tuning Result

In [ ]:
eval_results["transfer_difficulty"] = pd.Categorical(
    eval_results.name.replace({
        "qe_train_easy": "Easy",
        "qe_train_medium": "Medium",
        "qe_train_hard": "Hard",
    }),
    ["Easy", "Medium", "Hard"]
)
eval_results

In [ ]:
pdf = eval_results.melt(
    id_vars=["name", "transfer_difficulty"],
    value_vars=["average_precision_test", "average_precision_store"],
    var_name="dataset",
    value_name="average_precision",
)
pdf["dataset"] = pd.Categorical(
    pdf.dataset.replace({
        "average_precision_test": "Test",
        "average_precision_store": "Store",
    }),
    ["Test", "Store"]
)
pdf

In [ ]:
p = (
    ggplot(
        pdf,
        aes(
            x="average_precision",
            y="transfer_difficulty",
            color="dataset",
            shape="dataset",
        )
    )
    + geom_point(size=3)
    + labs(
        x="Average Precision", y="Transfer Difficulty", color="Dataset", shape="Dataset"
    )
    + theme_bw()
    + theme(aspect_ratio=1)
)
p.save(PROJECT_DIR / "plots" / "qe_experiment_average_precision.png")
p

## Plot Demo Result

In [ ]:
eval_results["dataset1"] = eval_results.name.apply(
    lambda x: x.replace("train", "test")
)
results = pd.merge(
    metrics_results, eval_results, left_on="dataset1", right_on="dataset1"
)
results

In [ ]:
p = (
    ggplot(
        results,
        aes(
            y="difference",
            x="mmd",
            color="transfer_difficulty",
            shape="transfer_difficulty",
        )
    )
    + geom_point(size=3)
    + labs(
        x="MMD",
        y="Difference in Average Precision (Test - Store)",
        color="Transfer Difficulty",
        shape="Transfer Difficulty",
    )
    + theme_bw()
    + theme(aspect_ratio=1)
)
p.save(PROJECT_DIR / "plots" / "qe_experiment_mmd.png")
p